In [ ]:
# ============================================================
# Project: NCR Ride Booking — Exploratory Data Analysis (EDA)
# Author : Vila Chung
# Purpose: Visualise ride patterns, compute key metrics, and
#          run statistical tests on the cleaned dataset.
# ============================================================

# ── Cell 1: Imports & Style Settings ────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

sns.set(style="whitegrid")
plt.rcParams["axes.unicode_minus"] = False   # fix minus sign rendering

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)

print("EDA environment ready.")

In [ ]:
# ── Cell 2: Load Cleaned Data ────────────────────────────────
CLEANED_FILE = "cleaned_ncr_rides.csv.gz"   # ← update path if needed

try:
    df = pd.read_csv(CLEANED_FILE, parse_dates=["Datetime", "Date_only"])
    print(f"Loaded — shape: {df.shape}")
    display(df.head(3))
except FileNotFoundError:
    raise FileNotFoundError(
        f"'{CLEANED_FILE}' not found. "
        "Run 01_Data_Cleaning_and_Preparation.ipynb first."
    )

In [ ]:
# ── Cell 3: High-Level KPIs ──────────────────────────────────
total      = len(df)
completed  = (df["Cancel_Type"] == "Completed").sum()
comp_rate  = completed / total * 100
cancel_rate = 100 - comp_rate

print("=== Key Performance Indicators ===")
print(f"  Total bookings   : {total:,}")
print(f"  Completed        : {completed:,}  ({comp_rate:.1f}%)")
print(f"  Cancelled/Failed : {total - completed:,}  ({cancel_rate:.1f}%)")

cancel_dist = df["Cancel_Type"].value_counts().to_frame("Count")
cancel_dist["Share (%)"] = (cancel_dist["Count"] / total * 100).round(1)
print("\n=== Cancellation-Type Breakdown ===")
display(cancel_dist)

In [ ]:
# ── Cell 4: SQL-Based Analysis (via sqlite3) ─────────────────
# Purpose: Replicate and extend key EDA findings using SQL queries.
# This demonstrates ability to work with both DataFrame-based and
# SQL-based workflows — both are standard in industry data roles.

import sqlite3

# ── Create is_cancelled column if it doesn't exist yet ───────
# This column is normally created in Notebook 03, but we define
# it here so this cell works standalone in Notebook 02.
if "is_cancelled" not in df.columns:
    df["is_cancelled"] = (df["Cancel_Type"] != "Completed").astype(int)
    print("'is_cancelled' column created.\n")

# ── Fix: Normalise column names before loading into SQLite ───
# SQLite does not handle spaces or mixed-case column names well.
# We create a temporary copy with clean snake_case column names.
df_sql = df.copy()
df_sql.columns = (
    df_sql.columns
    .str.strip()           # remove leading/trailing spaces
    .str.lower()           # lowercase everything
    .str.replace(" ", "_") # replace spaces with underscores
    .str.replace(r"[^\w]", "_", regex=True)  # replace any other special chars
)

# Confirm the new column names
print("Normalised column names:")
print(df_sql.columns.tolist(), "\n")

# Load into in-memory SQLite database
conn = sqlite3.connect(":memory:")
df_sql.to_sql("rides", conn, index=False, if_exists="replace")
print(f"SQLite table 'rides' ready — {len(df_sql):,} rows loaded.\n")


# ── Query 1: Cancellation rate by vehicle type ───────────────
# Business question: Which vehicle types have the worst completion rates?
q1 = """
SELECT   vehicle_type,
         COUNT(*)                           AS total_bookings,
         SUM(is_cancelled)                  AS cancelled,
         ROUND(AVG(is_cancelled) * 100, 1) AS cancel_rate_pct
FROM     rides
GROUP BY vehicle_type
ORDER BY cancel_rate_pct DESC
"""
print("=== Q1: Cancellation Rate by Vehicle Type ===")
display(pd.read_sql(q1, conn))


# ── Query 2: Peak-hour booking volume and cancel rate ────────
# Business question: When during the day do cancellations spike?
q2 = """
SELECT   hour,
         COUNT(*)                           AS total_bookings,
         ROUND(AVG(is_cancelled) * 100, 1) AS cancel_rate_pct
FROM     rides
GROUP BY hour
ORDER BY hour
"""
print("\n=== Q2: Hourly Volume & Cancellation Rate ===")
display(pd.read_sql(q2, conn))


# ── Query 3: Top 10 highest-risk pickup locations ────────────
# Business question: Which pickup zones have the most cancellations?
q3 = """
SELECT   pickup_location,
         COUNT(*)                           AS total_bookings,
         ROUND(AVG(is_cancelled) * 100, 1) AS cancel_rate_pct
FROM     rides
GROUP BY pickup_location
HAVING   COUNT(*) > 200
ORDER BY cancel_rate_pct DESC
LIMIT    10
"""
print("\n=== Q3: Top 10 Highest-Risk Pickup Locations ===")
display(pd.read_sql(q3, conn))


# ── Query 4: Weekend vs Weekday comparison ───────────────────
# Business question: Does weekend demand behave differently?
q4 = """
SELECT   CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
         COUNT(*)                           AS total_bookings,
         ROUND(AVG(booking_value), 2)       AS avg_booking_value,
         ROUND(AVG(ride_distance), 2)       AS avg_distance_km,
         ROUND(AVG(is_cancelled) * 100, 1) AS cancel_rate_pct
FROM     rides
GROUP BY is_weekend
"""
print("\n=== Q4: Weekend vs Weekday Behaviour ===")
display(pd.read_sql(q4, conn))


# ── Query 5: Payment method breakdown ───────────────────────
# Business question: Do certain payment methods correlate with cancellations?
q5 = """
SELECT   payment_method,
         COUNT(*)                           AS total_bookings,
         ROUND(AVG(is_cancelled) * 100, 1) AS cancel_rate_pct,
         ROUND(AVG(booking_value), 2)       AS avg_booking_value
FROM     rides
WHERE    payment_method IS NOT NULL
GROUP BY payment_method
ORDER BY total_bookings DESC
"""
print("\n=== Q5: Cancellation Rate by Payment Method ===")
display(pd.read_sql(q5, conn))


# Close connection
conn.close()
print("\nSQL analysis complete. Connection closed.")

In [ ]:
# ── Cell 5: Visualisation 1 — Booking Status Distribution ───
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

order = df["Cancel_Type"].value_counts().index
sns.countplot(data=df, x="Cancel_Type", order=order, ax=axes[0])
axes[0].set_title("Booking Status Distribution")
axes[0].set_xlabel("Status")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

axes[1].pie(
    [comp_rate, cancel_rate],
    labels=["Completed", "Cancelled / Failed"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#66c2a5", "#fc8d62"],
)
axes[1].set_title("Overall Completion Rate")

plt.tight_layout()
plt.savefig("02_booking_status.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 6: Visualisation 2 — Completion Rate Heatmap ───────
# Bin hours into labelled time periods
HOUR_BINS   = [0, 6, 9, 12, 15, 18, 21, 24]
HOUR_LABELS = ["Late Night", "AM Peak", "Morning",
               "Noon", "Afternoon", "PM Peak", "Evening"]

df["Time_Period"] = pd.cut(
    df["Hour"], bins=HOUR_BINS, labels=HOUR_LABELS, right=False
)

heatmap_data = df.pivot_table(
    index="Vehicle Type",
    columns="Time_Period",
    values="Cancel_Type",
    aggfunc=lambda x: (x == "Completed").mean() * 100,
).round(1)

plt.figure(figsize=(13, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.5)
plt.title("Completion Rate (%) by Vehicle Type × Time Period")
plt.xlabel("Time Period")
plt.ylabel("Vehicle Type")
plt.tight_layout()
plt.savefig("02_completion_heatmap.png", dpi=150)
plt.show()

In [ ]:

# ── Cell 7: Visualisation 3 — Hourly Volume & Cancel Rate ───
hourly_vol    = df.groupby("Hour").size().rename("Volume")
hourly_cancel = (
    df.groupby("Hour")["Cancel_Type"]
    .apply(lambda x: (x != "Completed").mean() * 100)
    .rename("Cancel Rate (%)")
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=hourly_vol.index, y=hourly_vol.values,
    name="Booking Volume", marker_color="steelblue", opacity=0.7,
))
fig.add_trace(go.Scatter(
    x=hourly_cancel.index, y=hourly_cancel.values,
    name="Cancel Rate (%)", mode="lines+markers",
    marker_color="coral", yaxis="y2",
))

fig.update_layout(
    title="Hourly Booking Volume vs. Cancellation Rate",
    title_x=0.5,
    xaxis_title="Hour of Day (0–23)",
    yaxis=dict(title="Booking Volume"),
    yaxis2=dict(title="Cancellation Rate (%)", overlaying="y", side="right"),
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
)
fig.write_html("02_hourly_trends.html")   # save interactive chart
fig.show()

In [ ]:
# ── Cell 8: Statistical Test — VTAT Difference by Vehicle Type
# Welch's t-test (does not assume equal variance — more robust)
TYPE_A, TYPE_B = "Go Sedan", "Auto"

vtat_a = df[df["Vehicle Type"] == TYPE_A]["Avg VTAT"].dropna()
vtat_b = df[df["Vehicle Type"] == TYPE_B]["Avg VTAT"].dropna()

if len(vtat_a) > 1 and len(vtat_b) > 1:
    t_stat, p_val = stats.ttest_ind(vtat_a, vtat_b, equal_var=False)
    print(f"Welch's t-test: {TYPE_A} vs {TYPE_B} (Avg VTAT)")
    print(f"  Mean {TYPE_A}: {vtat_a.mean():.2f} min  |  Mean {TYPE_B}: {vtat_b.mean():.2f} min")
    print(f"  t = {t_stat:.3f},  p = {p_val:.6f}")
    conclusion = "significant difference" if p_val < 0.05 else "no significant difference"
    print(f"  → At α=0.05, there is {conclusion} between the two groups.")
else:
    print("Insufficient data for t-test.")

In [ ]:
# ── Cell 9: Correlation Heatmap ─────────────────────────────
NUM_COLS = [
    "Avg VTAT", "Avg CTAT", "Ride Distance",
    "Booking Value", "Driver Ratings", "Customer Rating",
]

corr = df[NUM_COLS].corr().round(2)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, center=0,
            linewidths=0.5)
plt.title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.savefig("02_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 10: Export Summary for Dashboard / Tableau ──────────
vehicle_completion = (
    df.groupby("Vehicle Type")["Cancel_Type"]
    .apply(lambda x: (x == "Completed").mean() * 100)
    .round(1)
    .rename("Completion Rate (%)")
    .reset_index()
)

vehicle_completion.to_csv("vehicle_completion_rates.csv", index=False)
print("Saved → vehicle_completion_rates.csv  (ready for Tableau / Power BI)")
display(vehicle_completion)